In [1]:
import sys
# install into the SAME interpreter the notebook kernel uses, then restart the kernel afterwards
!{sys.executable} -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'calamari-ocr_calamari',
    'pdclab_peregrine',
    'juliaopt_pajarito.jl',
    'mic-dkfz_batchgenerators',
    'romerogroup_pyprocar',
    'jean-romain_lidr',
    'google-research_football',
    'drostlab_philentropy',
    'contextlab_hypertools',
    'lima1_purecn',
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,calamari-ocr_calamari


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  5%|████▉                                                                                        | 73/1367 [01:14<21:59,  1.02s/it]

Got Errors {'d9955f0a9539248fe7b6fb436f85b860caca5ddb': 'Key d9955f0a9539248fe7b6fb436f85b860caca5ddb not found in /da5_fast/All.sha1c/commit_89.tch'}


  7%|██████▌                                                                                      | 96/1367 [01:37<21:36,  1.02s/it]

Got Errors {'01308458a81e15bd2f1fe817a3b70eb153dd694b': 'Key 01308458a81e15bd2f1fe817a3b70eb153dd694b not found in /da5_fast/All.sha1c/commit_1.tch'}


  7%|██████▋                                                                                      | 99/1367 [01:40<21:34,  1.02s/it]

Got Errors {'06e82d48978bed3f090ff58ac822bb97b8449833': 'Key 06e82d48978bed3f090ff58ac822bb97b8449833 not found in /da5_fast/All.sha1c/commit_6.tch'}


... (80 outputs left out)

 30%|███████████████████████████▋                                                                | 411/1367 [06:59<16:15,  1.02s/it]

Got Errors {'f63f2488e687ff69122aab5b4036cbb25dc37759': 'Key f63f2488e687ff69122aab5b4036cbb25dc37759 not found in /da5_fast/All.sha1c/commit_118.tch'}


100%|███████████████████████████████████████████████████████████████████████████████████████████| 1367/1367 [23:13<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,5d61ae325d2faa67e821c444d50ddf083701d138,[6e5d92ad755ca34379752c54da5326ba6ef5c6d8],Christoph Wick <wick.chr.info@gmail.com>,1536677973,+0200,Christoph Wick <wick.chr.info@gmail.com>,1536677973,+0200,added support for abbyy.xml files as input (ne...,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,calamari-ocr_calamari
1,017766a18c260883f87747dfd3ee061dd36179d7,970959e781b1aa842752600c3e27544f06d8f8b4,[1da5d4df5938ea596caeab86afea8ef822793f37],Christoph Wick <wick.chr.info@gmail.com>,1571820729,+0200,Christoph Wick <wick.chr.info@gmail.com>,1571820729,+0200,Ignoring NaN as CER in early stopping\n,017766a18c260883f87747dfd3ee061dd36179d7,calamari-ocr_calamari


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0123fe4d1694365ffacff2d0ae66b913acf5e0bb', 'tree': '5d61ae325d2faa67e821c444d50ddf083701d138', 'parent': ['6e5d92ad755ca34379752c54da5326ba6ef5c6d8'], 'author': 'Christoph Wick <wick.chr.info@gmail.com>', 'author_time': 1536677973, 'author_tz': '+0200', 'committer': 'Christoph Wick <wick.chr.info@gmail.com>', 'committer_time': 1536677973, 'committer_tz': '+0200', 'message': 'added support for abbyy.xml files as input (new --dataset arg to switch to ABBYY or FILE=default)\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,lima1_purecn,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,Christoph Wick <wick.chr.info@gmail.com>,1536677973,added support for abbyy.xml files as input (ne...


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid = 'mmariaru'

dfinf = pd.DataFrame({
    'project_wocid': df_commit_data['project'],
    'commit_sha1':   df_commit_data['commit'],
    'author':        df_commit_data['author'],
    'time':          df_commit_data['author_time'],
    'commit message': df_commit_data['message'],
})

# sanity check: every one of the 10 projects should have rows here
print(dfinf['project_wocid'].value_counts())

# mode='w' (overwrite) so re-running the notebook doesn't duplicate rows; header included
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';', mode='w', header=True)
dfinf.head(3)

project_wocid
jean-romain_lidr            4899
contextlab_hypertools       1999
lima1_purecn                1375
romerogroup_pyprocar        1238
juliaopt_pajarito.jl        1191
drostlab_philentropy         890
calamari-ocr_calamari        850
mic-dkfz_batchgenerators     731
google-research_football     347
pdclab_peregrine             100
Name: count, dtype: int64


,project_wocid,commit_sha1,author,time,commit message
0,calamari-ocr_calamari,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,Christoph Wick <wick.chr.info@gmail.com>,1536677973,added support for abbyy.xml files as input (ne...
1,calamari-ocr_calamari,017766a18c260883f87747dfd3ee061dd36179d7,Christoph Wick <wick.chr.info@gmail.com>,1571820729,Ignoring NaN as CER in early stopping\n
2,calamari-ocr_calamari,02d46bf03a7371beae462e41f6f5b6d42151566c,Christoph Wick <wick.chr.info@gmail.com>,1571995296,Added training callback to allow external prog...


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [8]:
d = df_commit_data.copy()
d['dt'] = pd.to_datetime(d['author_time'], unit='s', utc=True)
woc_stats = (d.groupby('project')
               .agg(num_commits=('commit', 'nunique'),
                    num_authors=('author', 'nunique'),
                    min_time_unix=('author_time', 'min'),
                    max_time_unix=('author_time', 'max'))
               .reset_index())
woc_stats['min_time_utc'] = pd.to_datetime(woc_stats['min_time_unix'], unit='s', utc=True)
woc_stats['max_time_utc'] = pd.to_datetime(woc_stats['max_time_unix'], unit='s', utc=True)
woc_stats.to_csv('woc_stats.csv', index=False)
woc_stats

,project,num_commits,num_authors,min_time_unix,max_time_unix,min_time_utc,max_time_utc
0,calamari-ocr_calamari,850,48,1521559350,1749469622,2018-03-20 15:22:30+00:00,2025-06-09 11:47:02+00:00
1,contextlab_hypertools,1999,43,1469027405,1769667130,2016-07-20 15:10:05+00:00,2026-01-29 06:12:10+00:00
2,drostlab_philentropy,890,11,1424096347,1762174389,2015-02-16 14:19:07+00:00,2025-11-03 12:53:09+00:00
3,google-research_football,347,78,1559318565,1769902631,2019-05-31 16:02:45+00:00,2026-01-31 23:37:11+00:00
4,jean-romain_lidr,4899,36,1455710102,1759947297,2016-02-17 11:55:02+00:00,2025-10-08 18:14:57+00:00
5,juliaopt_pajarito.jl,1191,17,1456860843,1581804086,2016-03-01 19:34:03+00:00,2020-02-15 22:01:26+00:00
6,lima1_purecn,1375,28,1460665299,1771532264,2016-04-14 20:21:39+00:00,2026-02-19 20:17:44+00:00
7,mic-dkfz_batchgenerators,731,59,1489747027,1758661078,2017-03-17 10:37:07+00:00,2025-09-23 20:57:58+00:00
8,pdclab_peregrine,100,6,1586109630,1741800437,2020-04-05 18:00:30+00:00,2025-03-12 17:27:17+00:00
9,romerogroup_pyprocar,1238,37,1521992052,1762090662,2018-03-25 15:34:12+00:00,2025-11-02 13:37:42+00:00


In [9]:
import httpx

# GitHub repos exactly as given for this assignment
gh_repos = {
    'calamari-ocr_calamari':    'Calamari-OCR/calamari',
    'pdclab_peregrine':         'pdclab/peregrine',
    'juliaopt_pajarito.jl':     'JuliaOpt/Pajarito.jl',
    'mic-dkfz_batchgenerators': 'MIC-DKFZ/batchgenerators',
    'romerogroup_pyprocar':     'romerogroup/pyprocar',
    'jean-romain_lidr':         'Jean-Romain/lidR',
    'google-research_football': 'google-research/football',
    'drostlab_philentropy':     'drostlab/philentropy',
    'contextlab_hypertools':    'ContextLab/hypertools',
    'lima1_purecn':             'lima1/PureCN',
}

rows = []
with httpx.Client(follow_redirects=True, timeout=30,
                   headers={'Accept': 'application/vnd.github+json'}) as c:
    for wid, repo in gh_repos.items():
        info = c.get(f'https://api.github.com/repos/{repo}').json()
        last = c.get(f'https://api.github.com/repos/{repo}/commits', params={'per_page': 1}).json()
        if 'message' in info:  # API error, e.g. rate limit or not found
            print(repo, '->', info['message'])
            continue
        rows.append({
            'project_wocid': wid,
            'repo': repo,
            'stars': info.get('stargazers_count'),
            'forks': info.get('forks_count'),
            'last_commit_date': last[0]['commit']['committer']['date'][:10] if isinstance(last, list) and last else None,
        })

gh_stats = pd.DataFrame(rows)
gh_stats.to_csv('github_stats.csv', index=False)
gh_stats

,project_wocid,repo,stars,forks,last_commit_date
0,calamari-ocr_calamari,Calamari-OCR/calamari,1199,210,2026-05-13
1,pdclab_peregrine,pdclab/peregrine,208,38,2025-03-12
2,juliaopt_pajarito.jl,JuliaOpt/Pajarito.jl,141,21,2026-09-17
3,mic-dkfz_batchgenerators,MIC-DKFZ/batchgenerators,1173,231,2026-09-22
4,romerogroup_pyprocar,romerogroup/pyprocar,224,86,2025-09-20
5,jean-romain_lidr,Jean-Romain/lidR,712,141,2026-09-21
6,google-research_football,google-research/football,3663,1384,2025-06-17
7,drostlab_philentropy,drostlab/philentropy,150,22,2025-12-14
8,contextlab_hypertools,ContextLab/hypertools,1889,164,2026-09-05
9,lima1_purecn,lima1/PureCN,150,33,2026-02-19
